[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/solutions/part_03_engineering_workflow/ch15_git_clean_code/02_guided_todos_SOLUTION.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `ANNOTATED SOLUTION NOTEBOOK`


# Chapter 15 Guided To-Do Practice Labs

**Book Reference:** Chapter 15 (Part III Capstone), Section 15.5 (Guided To-Do Practice)  
Run **Step 0** to activate `sharukh-tutor`, build all 6 Git hygiene, secret-scanning, structured logging, and SHA-256 dataset provenance helpers, and pass each **function name (without parentheses!)** to `tutor.check(...)`. Every file/logging test runs inside an isolated `tempfile.TemporaryDirectory()`. Use `tutor.hint("ch15_todo_XX", level=1..3)` whenever you want a progressive nudge.

In [ ]:
# @title Step 0: Click Play (▶) to Activate Interactive Hints & Auto-Checker
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import sharukh_tutor as tutor

tutor.activate(chapter="ch15", notebook="guided_todos")


---
## To-Do 15.1 (`.gitignore` Auditing): Check Unignored Files

Write `check_gitignore_coverage(gitignore_text, filenames)` that parses non-blank, non-`#` comment pattern lines from `gitignore_text`, tests each filename using `fnmatch.fnmatch`, and **returns** a **sorted** `list[str]` of filenames that are **NOT** ignored.

In [ ]:
import fnmatch

def check_gitignore_coverage(gitignore_text, filenames):
    """Return sorted list of filenames NOT matched by .gitignore."""
    patterns = [
        line.strip()
        for line in gitignore_text.splitlines()
        if line.strip() and not line.strip().startswith("#")
    ]
    unignored = [
        fname for fname in filenames
        if not any(fnmatch.fnmatch(fname, pat) for pat in patterns)
    ]
    return sorted(unignored)


In [ ]:
tutor.check("ch15_todo_01", check_gitignore_coverage)


---
## To-Do 15.2 (Git Hygiene): Validate Commit Message Subject Line

Write `validate_commit_message(message)` that inspects **only the first subject line** (stripped) and verifies:
- It is non-empty.
- Its length satisfies `5 <= len(subject) <= 72`.
- It does **not** end with a period (`"."`).

**Return** a tuple `(is_valid: bool, reason: str)` with a clear diagnostic message.

In [ ]:
def validate_commit_message(message):
    """Validate first commit subject line; return (is_valid, reason)."""
    lines = message.splitlines()
    subject = lines[0].strip() if lines else ""
    if not subject:
        return (False, "Subject line cannot be empty")
    if len(subject) < 5:
        return (False, "Subject line must be at least 5 characters")
    if len(subject) > 72:
        return (False, "Subject line must be at most 72 characters")
    if subject.endswith("."):
        return (False, "Subject line should not end with a period")
    return (True, "Valid commit subject")


In [ ]:
tutor.check("ch15_todo_02", validate_commit_message)


---
## To-Do 15.3 (Pre-Commit Security): Scan Text for Leaked Secrets

Write `scan_for_leaked_secrets(text)` that scans lines **1-indexed** for the sensitive patterns `"ghp_"`, `"AIza"`, `"sk-"`, `"API_KEY="`, and `"PASSWORD="` (**case-sensitive**), and **returns** a sorted `list[int]` of matching 1-based line numbers (each line listed at most once).

In [ ]:
def scan_for_leaked_secrets(text):
    """Return 1-based line numbers containing sensitive secret tokens."""
    patterns = ("ghp_", "AIza", "sk-", "API_KEY=", "PASSWORD=")
    return [
        line_no
        for line_no, line in enumerate(text.splitlines(), start=1)
        if any(pat in line for pat in patterns)
    ]


In [ ]:
tutor.check("ch15_todo_03", scan_for_leaked_secrets)


---
## To-Do 15.4 (Structured Logging): Append a Formatted Log Line

Write `write_structured_log(log_path, level, message)` that appends a single UTF-8 log line formatted as `"[LEVEL] message"` (with uppercase level name) using `logging.FileHandler` (closed and removed in a `finally:` block so repeated calls never duplicate lines or lock temporary folders).

In [ ]:
import logging

_LOG_LEVELS = {
    "DEBUG": logging.DEBUG,
    "INFO": logging.INFO,
    "WARNING": logging.WARNING,
    "ERROR": logging.ERROR,
    "CRITICAL": logging.CRITICAL,
}

def write_structured_log(log_path, level, message):
    """Append '[LEVEL] message' to log_path via a closed FileHandler."""
    level_name = level.strip().upper()
    if level_name not in _LOG_LEVELS:
        raise ValueError(f"Unknown log level: {level!r}")
    logger = logging.getLogger("ch15_todo_logger")
    logger.setLevel(logging.DEBUG)
    logger.propagate = False
    handler = logging.FileHandler(log_path, mode="a", encoding="utf-8")
    handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
    logger.addHandler(handler)
    try:
        logger.log(_LOG_LEVELS[level_name], message)
    finally:
        logger.removeHandler(handler)
        handler.close()


In [ ]:
tutor.check("ch15_todo_04", write_structured_log)


---
## To-Do 15.5 (Cryptographic Integrity): Compute File SHA-256 Digest

Write `compute_file_sha256(filepath)` that opens `filepath` in binary mode (`"rb"`), streams bytes into `hashlib.sha256()`, and **returns** the lowercase 64-character hex digest string.

In [ ]:
import hashlib

def compute_file_sha256(filepath):
    """Return lowercase 64-char SHA-256 hex digest of filepath."""
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    return hasher.hexdigest()


In [ ]:
tutor.check("ch15_todo_05", compute_file_sha256)


---
## To-Do 15.6 (Dataset Provenance): Build a Sorted Dataset Manifest

Write `build_dataset_manifest(data_dir)` that scans regular `.csv` and `.json` files directly inside `data_dir`, sorted alphabetically by filename, and **returns** a dictionary mapping `filename -> {"bytes": int, "sha256": str}` (ignoring non-data files and subdirectories).

In [ ]:
import hashlib
from pathlib import Path

def build_dataset_manifest(data_dir):
    """Map sorted .csv/.json filenames to byte size and SHA-256."""
    manifest = {}
    for p in sorted(Path(data_dir).iterdir(), key=lambda x: x.name):
        if p.is_file() and p.suffix.lower() in {".csv", ".json"}:
            digest = hashlib.sha256(p.read_bytes()).hexdigest()
            manifest[p.name] = {
                "bytes": p.stat().st_size,
                "sha256": digest,
            }
    return manifest


In [ ]:
tutor.check("ch15_todo_06", build_dataset_manifest)


---
## Annotated Debugging Post-Mortem (Solution Notebook Only)

- **To-Do 15.1 & 15.2 (`.gitignore` Negation & Multi-Line Commits):** `check_gitignore_coverage` returns **UNIGNORED** files (the ones Git *would* commit), so always negate `any(fnmatch.fnmatch(...))` and skip `#` comments first. In `validate_commit_message`, inspect only `splitlines()[0].strip()` so multi-line commit bodies are never penalized for length or punctuation.
- **To-Do 15.3 & 15.4 (1-Based Line Numbers & Handler Leaks):** Start `enumerate(..., start=1)` to match editor line numbers, and use `any(...)` so a line with two secret tokens is only reported once. For `logging.FileHandler`, always call `logger.removeHandler(handler)` and `handler.close()` in a `finally:` block so subsequent calls never duplicate lines or lock temporary folders.
- **To-Do 15.5 & 15.6 (Binary Mode `'rb'` & `is_file()` Guard):** `hashlib.sha256()` hashes raw `bytes`, never decoded `str`—opening files in `'rb'` avoids both `TypeError` and `UnicodeDecodeError` on binary assets. Always check `p.is_file()` before `p.stat()` or `p.read_bytes()` so a folder named `archive.csv/` never triggers `IsADirectoryError`.

In [ ]:
tutor.summary()
